# 03 LeRobot + MuJoCo 仿真与 Policy 训练

目标：理解具身智能完整闭环：

observation → policy → action → environment → next observation

重点理解：
- MuJoCo如何实现 f(s,a)
- Gymnasium环境接口
- LeRobot数据与仿真的关系
- Policy训练和部署流程


In [2]:
import sys
import torch
import mujoco
import gymnasium as gym

print(sys.executable)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
print('mujoco:', mujoco.__version__)
print('gym:', gym.__version__)

d:\Desktop\robot\envs\lerobot-win\python.exe
torch: 2.11.0+cu128
cuda: True
mujoco: 3.11.0
gym: 1.3.0


## 1. MuJoCo中的状态转移

机器人环境定义了：

s(t+1)=f(s(t),a(t))

LeRobot数据集保存历史轨迹，但不负责计算f。
MuJoCo物理引擎根据机器人模型和动力学计算下一状态。

In [3]:
env = gym.make('HalfCheetah-v5')
obs, info = env.reset()

print('observation shape:', obs.shape)
print('action space:', env.action_space)

observation shape: (17,)
action space: Box(-1.0, 1.0, (6,), float32)


## 2. env.step(action)

输入动作：

action_t

环境返回：

observation_(t+1), reward, done, info

这就是机器人控制循环。

In [4]:
action = env.action_space.sample()
next_obs, reward, terminated, truncated, info = env.step(action)

print('action:', action)
print('next state:', next_obs[:5])
print('reward:', reward)

action: [ 0.45889768 -0.31684417 -0.62170416  0.18132065 -0.5343376  -0.81517947]
next state: [ 0.05431795  0.01628102  0.14447783 -0.16554737 -0.25907675]
reward: 0.007878806822670281


## 3. 随机Policy

随机动作不是AI，只用于验证环境。

真正policy：

action = π(observation)


In [5]:
obs, info = env.reset()
for i in range(10):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(i, reward)

0 0.18623697014875013
1 -0.9555192550856417
2 -1.1942116231521467
3 -0.9118468852540766
4 -0.1914891509321277
5 0.5188097666077315
6 0.7747823884494975
7 0.24609518172382605
8 -0.8582457779443833
9 -0.2766647004488202


## 4. LeRobot与MuJoCo关系

LeRobot Dataset：
- 专家示范
- observation
- action

MuJoCo：
- 机器人模型
- 动力学
- 碰撞
- 状态更新

Policy训练学习：

observation → action

## 5. ACT / Diffusion Policy训练流程

batch数据：

observation + expert action

模型预测action

计算loss

更新参数

## 下一步

1. 下载PushT环境
2. 将LeRobot PushT dataset连接环境
3. 学习ACT代码结构
4. 完成policy推理闭环